# 01 - Clean Crop Production (Silver)

`bronze.crop_production` -> `silver.crop_production`

1. Trim text columns and upper-case `crop_group`
2. Combine duplicate rows for the same province + crop + year + quarter (production is summed)
3. Round numbers
4. Recalculate lags, rolling anomalies, YoY growth and climate stress **after** combining duplicates
5. Write `silver.crop_production`

**Gap-safe lags:** some province + crop series have missing quarters. A lag is only filled when the earlier
row is really `n` quarters back (`time_index` differs by exactly `n`). Otherwise it stays `NULL`
(a missing value is not the same as zero production).

## 1. Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from common.delta_io import write_delta_table
from configs.etl_config import (
    BRONZE_CROP_TABLE,
    SILVER_CROP_TABLE,
    CROP_KEY_COLUMNS,
)


print(f"Source: {BRONZE_CROP_TABLE}")
print(f"Target: {SILVER_CROP_TABLE}")

## 2. Load Bronze and standardize text

In [0]:
raw_df = spark.table(BRONZE_CROP_TABLE)
print(f"Bronze rows: {raw_df.count():,}")

standardized_df = (
    raw_df
    .withColumn("province", F.trim(F.col("province")))
    .withColumn("crop", F.trim(F.col("crop")))
    .withColumn("quarter", F.trim(F.col("quarter")))
    .withColumn("crop_group", F.upper(F.trim(F.col("crop_group"))))
    .withColumn("risk_label", F.trim(F.col("risk_label")))
)

## 3. Combine duplicate business keys

Production is summed. Climate columns repeat on every duplicate row, so we keep the first non-null value.
The old lag / growth / stress columns are dropped here because step 5 recalculates them.

In [0]:
columns_to_keep = [
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "precipitation_sum", "rain_sum", "precipitation_hours",
    "sunshine_duration", "shortwave_radiation_sum", "reference_evapotranspiration_mm",
    "wind_speed_10m_max", "wind_gusts_10m_max",
    "rain_normal", "temp_normal", "rainfall_deviation_pct",
    "temperature_anomaly_c", "oni_index",
    "quarter_num", "time_index",
    "enso_phase_la_nina", "enso_phase_neutral",
    "crop_group", "risk_label",
]

aggregations = [F.sum("production").alias("production")]
aggregations += [F.first(column, ignorenulls=True).alias(column) for column in columns_to_keep]
aggregations += [
    F.max("_ingested_at").alias("_ingested_at"),
    F.first("_source_file").alias("_source_file"),
]

deduplicated_df = standardized_df.groupBy(*CROP_KEY_COLUMNS).agg(*aggregations)

## 4. Round numbers

In [0]:
rounding_rules = {
    0: ["sunshine_duration"],
    2: [
        "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
        "precipitation_sum", "rain_sum", "shortwave_radiation_sum",
        "reference_evapotranspiration_mm", "wind_speed_10m_max", "wind_gusts_10m_max",
        "rain_normal", "temp_normal", "rainfall_deviation_pct",
    ],
    3: ["temperature_anomaly_c", "oni_index", "production"],
}

rounded_df = deduplicated_df
for decimals, columns in rounding_rules.items():
    for column in columns:
        rounded_df = rounded_df.withColumn(column, F.round(F.col(column), decimals))

# precipitation_hours is a whole number of hours
rounded_df = rounded_df.withColumn("precipitation_hours", F.col("precipitation_hours").cast("int"))

## 5. Recalculate lags, rolling anomalies, YoY growth and climate stress

Each province + crop is one time series, ordered by `time_index`.

In [0]:
by_province_and_crop = Window.partitionBy("province", "crop").orderBy("time_index")
last_four_quarters = by_province_and_crop.rowsBetween(-3, 0)

features_df = rounded_df

# Production lags 1 to 4 quarters back (NULL when the quarter is missing from the series)
for quarters_back in (1, 2, 3, 4):
    earlier_production = F.lag("production", quarters_back).over(by_province_and_crop)
    earlier_time_index = F.lag("time_index", quarters_back).over(by_province_and_crop)
    features_df = features_df.withColumn(
        f"production_lag_{quarters_back}",
        F.when(F.col("time_index") - earlier_time_index == quarters_back, earlier_production),
    )

features_df = (
    features_df
    .withColumn(
        "temp_anomaly_rolling",
        F.round(F.avg("temperature_anomaly_c").over(last_four_quarters), 3),
    )
    .withColumn(
        "rain_anomaly_rolling",
        F.round(F.avg("rainfall_deviation_pct").over(last_four_quarters), 3),
    )
    # YoY growth = (this quarter - same quarter last year) / same quarter last year.
    # F.when() without otherwise() gives NULL when lag_4 is missing or zero.
    .withColumn(
        "growth_rate_yoy",
        F.when(
            F.col("production_lag_4") != 0,
            F.round((F.col("production") - F.col("production_lag_4")) / F.col("production_lag_4"), 4),
        ),
    )
    .withColumn(
        "climate_stress",
        F.round(F.abs(F.col("rain_anomaly_rolling")) + F.abs(F.col("temp_anomaly_rolling")), 3),
    )
)

## 6. Write the Silver table

In [0]:
write_delta_table(features_df, SILVER_CROP_TABLE)
print(f"Silver rows: {spark.table(SILVER_CROP_TABLE).count():,}")
print("Next: 02_clean_weather")